# MCP Server

TuiML includes an **MCP (Model Context Protocol) Server** that exposes 200+ ML tools that AI assistants like Claude can use directly.

## What You'll Learn

1. **MCP Server Setup** - Run the server
2. **Claude Desktop Integration** - Connect Claude to TuiML
3. **Quick Demo** - See it in action

### Verify MCP Availability

In [1]:
import tuiml
print(f"TuiML version: {tuiml.__version__}")

# Check MCP availability
from tuiml.llm import MCP_AVAILABLE, get_tool_count
print(f"MCP Available: {MCP_AVAILABLE}")
print(f"Total ML Tools: {get_tool_count()}")

TuiML version: 0.1.0
MCP Available: True
Total ML Tools: {'algorithm': 83, 'preprocessing': 77, 'dataset': 24, 'feature': 19, 'splitting': 14}


---

## 1. Running the MCP Server

The MCP server exposes TuiML tools via the Model Context Protocol.

### Start the Server

```bash
# Simplest way
tuiml-mcp

# Or using Python module
python -m tuiml.llm.server
```

The server starts and listens for MCP connections via stdio.

**Server Options:**
```bash
tuiml-mcp --help   # Show help
tuiml-mcp --info   # Show server info and tool count
```

### Available Tools

The MCP server exposes these workflow tools:

| Tool | Description |
|------|-------------|
| `tuiml_train` | Train ML models (classifiers, regressors, clusterers) |
| `tuiml_predict` | Make predictions with trained models |
| `tuiml_evaluate` | Evaluate model performance |
| `tuiml_experiment` | Compare multiple algorithms |
| `tuiml_list` | List available components |
| `tuiml_describe` | Get component details |
| `tuiml_search` | Search for components |
| `tuiml_upload_data` | Upload dataset content |

In [2]:
from tuiml.llm import get_workflow_tools

tools = get_workflow_tools()
print("MCP Workflow Tools:")
for name, schema in tools.items():
    print(f"  {name}")
    print(f"    {schema['description'][:80]}...")
    print()

MCP Workflow Tools:
  tuiml_train
    Train a machine learning model. Supports supervised (classifiers, regressors) an...

  tuiml_predict
    Make predictions using a trained model on new data. Supports supervised models, ...

  tuiml_evaluate
    Evaluate a trained model on test data and compute metrics....

  tuiml_experiment
    Compare multiple algorithms on one or more datasets with cross-validation and st...

  tuiml_upload_data
    Upload dataset content directly (CSV or ARFF text). Returns a file path that can...

  tuiml_save_model
    Copy a trained model to a custom path. Use this when the user wants to save or d...

  tuiml_serve_model
    Start a REST API server to serve a trained model for predictions. Returns the UR...

  tuiml_stop_server
    Stop a running model serving API server....

  tuiml_server_status
    Get status of running model serving API servers....

  tuiml_plot
    Generate a visualization/plot for model analysis. Returns the plot as an inline ...

  tu

---

## 2. Claude Desktop Integration

Connect Claude Desktop to TuiML MCP server.

### Step 1: Find Claude Config File

| OS | Path |
|----|------|
| macOS | `~/Library/Application Support/Claude/claude_desktop_config.json` |
| Windows | `%APPDATA%\Claude\claude_desktop_config.json` |
| Linux | `~/.config/Claude/claude_desktop_config.json` |

### Step 2: Add TuiML Server

Edit `claude_desktop_config.json`:

**Simplest (Recommended):**
```json
{
    "mcpServers": {
        "tuiml": {
            "command": "tuiml-mcp"
        }
    }
}
```

**Using Python module:**
```json
{
    "mcpServers": {
        "tuiml": {
            "command": "python",
            "args": ["-m", "tuiml.llm.server"]
        }
    }
}
```

**With specific Python path:**
```json
{
    "mcpServers": {
        "tuiml": {
            "command": "/path/to/your/venv/bin/tuiml-mcp"
        }
    }
}
```

### Step 3: Restart Claude Desktop

1. Quit Claude Desktop completely
2. Restart Claude Desktop
3. Look for the hammer icon (🔨) in the chat - this shows MCP tools are available

### Step 4: Test It!

Ask Claude:
- *"Train a RandomForestClassifier on the iris dataset"*
- *"Compare C45TreeClassifier and NaiveBayesClassifier on wine dataset"*
- *"List available clustering algorithms"*

---

## 3. Deployment Options

### Local Machine

```bash
# Configure Claude Desktop (see above)
# Claude starts the server automatically
```

### VPS / Cloud Server

```bash
# SSH into your server
ssh user@your-server.com

# Install TuiML
pip install tuiml

# Run server (for testing)
tuiml-mcp
```

**For persistent running, use systemd:**

```bash
# /etc/systemd/system/tuiml-mcp.service
[Unit]
Description=TuiML MCP Server
After=network.target

[Service]
Type=simple
User=your-user
WorkingDirectory=/home/your-user
ExecStart=/home/your-user/.local/bin/tuiml-mcp
Restart=always

[Install]
WantedBy=multi-user.target
```

```bash
sudo systemctl enable tuiml-mcp
sudo systemctl start tuiml-mcp
```

### Docker

```dockerfile
FROM python:3.11-slim
WORKDIR /app
RUN pip install --no-cache-dir tuiml
CMD ["tuiml-mcp"]
```

```bash
docker build -t tuiml-mcp .
docker run -it tuiml-mcp
```

---

## 4. Quick Demo (Without MCP)

You can also use TuiML tools programmatically without the MCP server.

In [3]:
from tuiml.llm import execute_tool

# Train a model
result = execute_tool(
    "tuiml_train",
    algorithm="RandomForestClassifier",
    data="iris",
    cv=5,
    algorithm_params={"n_estimators": 10, "random_state": 42}
)

print(f"Status: {result['status']}")
print(f"Model: {result.get('model_class')}")
print(f"Metrics: {result.get('metrics')}")

Status: success
Model: RandomForestClassifier
Metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9395583160800551, 'cv_f1_score_std': 0.04058971815950488}


In [4]:
# List classifiers
result = execute_tool(
    "tuiml_list",
    category="algorithm",
    limit=10
)

print(f"Found {result['total']} algorithms. First 10:")
for comp in result['components']:
    print(f"  - {comp['name']}")

Found 83 algorithms. First 10:
  - tuiml_algorithm_NaiveBayesClassifier
  - tuiml_algorithm_NaiveBayesMultinomialClassifier
  - tuiml_algorithm_BayesianNetworkClassifier
  - tuiml_algorithm_DecisionStumpClassifier
  - tuiml_algorithm_C45TreeClassifier
  - tuiml_algorithm_RandomTreeClassifier
  - tuiml_algorithm_RandomForestClassifier
  - tuiml_algorithm_ReducedErrorPruningTreeClassifier
  - tuiml_algorithm_HoeffdingTreeClassifier
  - tuiml_algorithm_LogisticModelTreeClassifier


In [5]:
# Compare algorithms
result = execute_tool(
    "tuiml_experiment",
    algorithms=["RandomForestClassifier", "NaiveBayesClassifier", "C45TreeClassifier"],
    data="iris",
    target="class",
    cv=5
)

print(f"Experiment Status: {result['status']}")
if result['status'] == 'success':
    for dataset, models in result.get('results', {}).items():
        print(f"\nDataset: {dataset}")
        for model, metrics in models.items():
            for metric_name, data in metrics.items():
                print(f"  {model}: {metric_name} = {data.get('mean', 'N/A'):.4f}")

Experiment Status: success

Dataset: iris
  RandomForestClassifier: accuracy_score = 0.9467
  NaiveBayesClassifier: accuracy_score = 0.9600
  C45TreeClassifier: accuracy_score = 0.9267
